Memória dos Agentes
# **Long Term com Gravação**
## Diogo Veiga - RM375392

In [24]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver

In [25]:
!pip install langchain-openai -q

In [26]:
import os
from google.colab import userdata
admin_api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = admin_api_key

model = init_chat_model("openai:gpt-4o-mini")

def tutor(state: MessagesState):

    return {"messages": [model.invoke(state["messages"])]}

builder = StateGraph(MessagesState)

builder.add_node("tutor", tutor)

builder.add_edge(START, "tutor")

graph = builder.compile(checkpointer=InMemorySaver())

In [37]:
import uuid
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore



store = InMemoryStore(index={"embed": "openai:text-embedding-3-small", "dims": 1536})



def tutormem(state: MessagesState, config: RunnableConfig, *, store: BaseStore):

  ns = ("memories", config["configurable"]["user_id"])

  pergunta = state["messages"][-1].content

  fatos = store.search(ns, query=pergunta, limit=3)   # LER​

  perfil = "\n".join(f.value["data"] for f in fatos)

  if "lembre" in pergunta.lower():                    # ESCREVER​

    store.put(ns, str(uuid.uuid4()), {"data": pergunta})

    sys = {"role": "system", "content": f"Perfil do aluno:\n{perfil}"}

    return {"messages": [model.invoke([sys, *state["messages"]])]}


graph = builder.compile(checkpointer=InMemorySaver(), store=store)

cfg = {"configurable": {"thread_id": "t1", "user_id": "ana"}}

In [38]:
# 1. Definindo o namespace que foi utilizado para salvar os dados
namespace = [
    ("memories", "amanda"),
    ("memories", "giulia"),
]

In [40]:
# Insere um dado passando (namespace, chave_única, dicionário_com_os_dados)
#store.put(("memories", "ana"), "user_preference", {"data": "A Ana gosta de cafe sem açucar"})

store.put(
    namespace[0],  # Amanda
    "shopping_cart",
    {
        "items": [
            "Notebook MacBook Air Rosa 512 GB",
            "Mouse sem fio Apple",
            "Capa para notebook"
        ]
    }
)

store.put(
    namespace[1],  # Giulia
    "shopping_cart",
    {
        "items": [
            "Tablet iPad Rosa 512 GB",
            "Fone de ouvido Bluetooth Bose",
            "Hub USB-C"
        ]
    }
)

In [41]:
for ns in namespace:
    carrinho = store.get(ns, "shopping_cart")

    print(f"\n=== Carrinho de {ns[1].title()} ===")

    if carrinho is not None:
        for item in carrinho.value["items"]:
            print(f"- {item}")
    else:
        print("Carrinho vazio.")


=== Carrinho de Amanda ===
- Notebook MacBook Air Rosa 512 GB
- Mouse sem fio Apple
- Capa para notebook

=== Carrinho de Giulia ===
- Tablet iPad Rosa 512 GB
- Fone de ouvido Bluetooth Bose
- Hub USB-C
